# **RAGAS Implementation**


Importing the Important Lib

In [ ]:
import os
import asyncio
import warnings
import pandas as pd
from  dotenv import load_dotenv
from openai import OpenAI

**Importing RAGAS Libraries**

In [ ]:
from ragas.llms import llm_factory
from ragas.embeddings import HuggingFaceEmbeddings
from ragas import SingleTurnSample
from ragas.metrics.collections import (
    Faithfulness,
    AnswerRelevancy,
    ContextPrecision,
    ContextRecall,
    AnswerCorrectness
)

**DeepEval Libraries**

In [ ]:

from deepeval.test_case import LLMTestCase , ToolCall
from deepeval.metrics import ToolCorrectnessMetric
from openai import AsyncOpenAI

**Loading APi Keys**

In [ ]:
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
RAGAS_JUDGE_MODEL = os.getenv("RAGAS_JUDGE_MODEL", "gpt-4o-mini")
HF_TOKEN = os.getenv("HF_TOKEN")
EVAL_SECRET = os.getenv("EVAL_SECRET")

def _mask_secret(value: str | None, prefix_len: int = 4) -> str:
    if not value:
        return "<missing>"
    if len(value) <= prefix_len * 2:
        return "<set>"
    return f"{value[:prefix_len]}...{value[-prefix_len:]} (len={len(value)})"

print("OPENAI_API_KEY:", _mask_secret(OPENAI_API_KEY))
print("RAGAS_JUDGE_MODEL:", RAGAS_JUDGE_MODEL)
print("HF_TOKEN:", _mask_secret(HF_TOKEN))
print("EVAL_SECRET:", _mask_secret(EVAL_SECRET))


**Enviorment Setup**

In [ ]:
client = AsyncOpenAI(api_key=OPENAI_API_KEY, base_url="https://api.openai.com/v1")
JudgeModel = llm_factory("gpt-4o-mini" , client=client , provider="openai")

ragas_embeddings  = HuggingFaceEmbeddings(
    model="sentence-transformers/all-MiniLM-L6-v2" ,
    use_api=False,
    api_key=HF_TOKEN,
    )


**Helper Function**

In [ ]:
import json
import textwrap

def print_eval_payload(payload: dict, faithfulness_result=None, max_context_chars=400):
    """Pretty-print /eval/run response for debugging RAGAS."""

    print("\n" + "=" * 72)
    print("  EVAL RUN PAYLOAD")
    print("=" * 72)

    print(f"\nQuestion : {payload.get('question', '')}")
    print(f"Answer   : {payload.get('answer', '')}")

    if faithfulness_result is not None:
        score = getattr(faithfulness_result, "value", faithfulness_result)
        print(f"\nFaithfulness : {score:.4f}")

    print("\n--- Latency ---")
    print(f"Retrieval  : {payload.get('retrieval_latency_ms')} ms")
    print(f"Generation : {payload.get('generation_latency_ms')} ms")

    print("\n--- Knowledge base ---")
    print(f"KB chunks        : {payload.get('kb_chunk_count')}")
    print(f"KB manifest hash : {payload.get('kb_manifest_hash')}")

    print("\n--- Retrieved sources ---")
    for i, meta in enumerate(payload.get("context_metadata") or [], start=1):
        print(
            f"  [{i}] {meta.get('file_name')} "
            f"(score={meta.get('score'):.4f})"
            if meta.get("score") is not None
            else f"  [{i}] {meta.get('file_name')}"
        )

    print(f"\n--- Contexts ({len(payload.get('contexts') or [])}) ---")
    for i, ctx in enumerate(payload.get("contexts") or [], start=1):
        cleaned = " ".join(ctx.split())  # collapse noisy newlines/spaces
        preview = cleaned[:max_context_chars]
        suffix = "..." if len(cleaned) > max_context_chars else ""
        print(f"\n[Context {i}]")
        print(textwrap.fill(preview + suffix, width=100))

    if payload.get("resource_links"):
        print("\n--- Resource links ---")
        for link in payload["resource_links"]:
            print(f"  - {link}")

    print("\n" + "=" * 72 + "\n")

# CoolDown Helper Function
async def async_cooldown(seconds=60):
    """Async cooldown - works inside Jupyter's running event loop."""
    print(f"\n⏳ Cooldown {seconds}s (Groq rate-limit buffer)...", end=" ")

    for _ in range(seconds // 10):
        await asyncio.sleep(10)
        print(".", end="", flush=True)

    print("  ✅ Ready.\n")


# Score Checker Function
def show_scores(df, col, title):
    """Colour-coded score table for one metric column."""

    print(f"\n{'=' * 60}")
    print(f"  {title}")
    print(f"{'=' * 60}")

    for i, row in df.iterrows():
        score = row.get(col, float("nan"))

        # Determine score status
        if score != score:  # NaN check
            bar = "⚪"
        elif score >= 0.75:
            bar = "🟢"
        elif score >= 0.5:
            bar = "🟡"
        else:
            bar = "🔴"

        # Get user input/question
        q = str(row.get("user_input", f"Sample {i + 1}"))[:55]

        print(f" {bar}  {score:.2f}  |  {q}")

    # Calculate average score
    avg = df[col].mean()

    # Determine overall label
    label = (
        "🟢 Good"
        if avg >= 0.75
        else ("🟡 Fair" if avg >= 0.5 else "🔴 Poor")
    )
    print(f"{'=' * 60}")
    print(f"  📊 Average: {avg:.2f}  |  {label}")
    print(f"{'=' * 60}\n")

**Metric Implemenetation**

In [ ]:

client = AsyncOpenAI(api_key=OPENAI_API_KEY)
JudgeModel = llm_factory("gpt-4o-mini", client=client, provider="openai")
FaithfulnessMetric = Faithfulness(llm=JudgeModel)

**Implementation With Static Data**

In [ ]:
question = "What is your degree?"
answer = "I have a B.Tech from MGM College, Noida."
contexts = [
    "EDUCATION: 2019~2023 Mahatma Gandhi Mission's College Of Engineering & Technology, Noida, NCR Bachelor of Technology : GPA: 8.13"
]

result = await FaithfulnessMetric.ascore(
    user_input=question,
    response=answer,
    retrieved_contexts=contexts,
)

print(f"Faithfulness score: {result.value:.4f}")


**API Data FaithFullness**

In [ ]:
import httpx
headers = {"X-Eval-Secret": os.getenv("EVAL_SECRET")}
payload = httpx.post(
    "http://127.0.0.1:8000/eval/run",
    headers=headers,
    json={"question": "What is your any architecture Degree?"},
).json()

# print(payload)

result = await FaithfulnessMetric.ascore(
    user_input=payload["question"],
    response=payload["answer"],
    retrieved_contexts=payload["contexts"],
)

print_eval_payload(payload, faithfulness_result=result)